# Module 3 — Coat-protein expression
### BruSafe / CAPSID · iGEM 2026

This notebook reproduces the two figures of the M3 panel:

- **(a)** mRNA and protein expression reaching steady state
- **(b)** the justification of the *D = P/2* approximation

The results are **model simulations with literature parameters**, not laboratory data. They are the *a priori* prediction (Mode A), that is, what is expected before measuring.

The cells are run in order (`Runtime → Run all` in Colab).

## 1 · Setup

In [ ]:
# ---------------------------------------------------------------
# Only numpy, scipy and matplotlib are used; all three are
# preinstalled in Colab, so nothing needs to be installed.
# ---------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

RT = 0.593        # kcal/mol at ~298 K
N_CAP = 90        # coat-protein dimers per T=3 capsid (Ni 1995)

## 2 · Parameters

All values come from the general *E. coli* literature; **they do not correspond to our strain or to our plasmid**. These are precisely the parameters that the laboratory team will have to measure.

In [ ]:
# Parameters: (central value, (min, max), source)
# The range is not a confidence interval, but the support of the prior taken
# from the literature. When the central values are changed, the figures are recomputed.
PARAMS = {
    "t_half_mrna_min":   (5.0,   (2.0, 8.0),   "Bernstein 2002; Selinger 2003"),
    "t_double_min":      (40.0,  (25., 60.),   "Milo 2010"),
    "alpha_tx_nM_min":   (6.0,   (1.0, 30.),   "Milo 2010"),
    "k_tl_permin":       (4.0,   (1.0, 12.),   "Milo 2010"),
    "delta_prot_permin": (0.002, (0.0, 0.01),  "Milo 2010"),
    "Kd_dim_uM":         (0.01,  (1e-3, 1.0),  "Ni 1995 (obligate dimer)"),
}

def central(params=PARAMS):
    """Extract the central values only."""
    return {k: v[0] for k, v in params.items()}

p = central()
for k, (v, rng, src) in PARAMS.items():
    print(f"{k:<20} {v:<8} range {str(rng):<14} {src}")

## 3 · The model

The model consists of two coupled ordinary differential equations (mRNA and protein) and the dimerisation equilibrium.

In [ ]:
def induction(t, t_on=30.0, ramp=10.0):
    """Smooth inducer switch (IPTG): 0 -> 1 around t_on."""
    return 1.0 / (1.0 + np.exp(-(t - t_on) / ramp))


def expression_ode(y, t, alpha, delta_m, mu, k_tl, delta_P, t_on):
    """dm/dt = alpha*u(t) - (delta_m + mu)*m
       dP/dt = 1e-3*k_tl*m - (delta_P + mu)*P

    m in nM, P in uM monomer-equivalents (1e-3 converts nM -> uM).

    Note: mu (dilution by growth) appears in both equations. For the coat
    protein, which is stable (delta_P = 0.002 /min), mu dominates the loss by
    a factor of ~9: the lifetime of the protein is set by cell division and
    not by proteolysis.
    """
    m, P = y
    u = induction(t, t_on)
    dm = alpha * u - (delta_m + mu) * m
    dP = 1e-3 * k_tl * m - (delta_P + mu) * P
    return [dm, dP]


def derived_rates(p):
    """Convert half-lives into first-order rates."""
    delta_m = np.log(2) / p["t_half_mrna_min"]
    mu      = np.log(2) / p["t_double_min"]
    return delta_m, mu


def simulate_expression(p, t_grid, t_on=30.0):
    """Integrate the ODEs. Returns (m in nM, P in uM)."""
    delta_m, mu = derived_rates(p)
    sol = odeint(expression_ode, [0.0, 0.0], t_grid,
                 args=(p["alpha_tx_nM_min"], delta_m, mu,
                       p["k_tl_permin"], p["delta_prot_permin"], t_on),
                 rtol=1e-8, atol=1e-10)
    return sol[:, 0], sol[:, 1]


def expression_steady_state(p):
    """Closed form with u -> 1:
         m_ss = alpha / (delta_m + mu)
         P_ss = 1e-3 * k_tl * m_ss / (delta_P + mu)
    It can be verified analytically."""
    delta_m, mu = derived_rates(p)
    m_ss = p["alpha_tx_nM_min"] / (delta_m + mu)
    P_ss = 1e-3 * p["k_tl_permin"] * m_ss / (p["delta_prot_permin"] + mu)
    return m_ss, P_ss


def monomer_dimer_split(P_tot_uM, Kd_dim_uM):
    """Equilibrium partition 2M <-> D:
         2D + M = P_tot ,  D = M^2 / Kd
    Solves (2/Kd)M^2 + M - P_tot = 0.

    When Kd -> 0 (strong dimer, Ni 1995), D -> P_tot/2. This is the
    approximation used by the main model; here it is verified instead of
    assumed."""
    a = 2.0 / Kd_dim_uM
    M = (-1.0 + np.sqrt(1.0 + 4.0 * a * P_tot_uM)) / (2.0 * a)
    D = M * M / Kd_dim_uM
    return M, D


# --- numerical check ---
delta_m, mu = derived_rates(p)
m_ss, P_ss = expression_steady_state(p)
M_eq, D_eq = monomer_dimer_split(P_ss, p["Kd_dim_uM"])

print(f"delta_m = {delta_m:.4f} /min     mu = {mu:.5f} /min")
print(f"protein loss: mu/delta_P = {mu/p['delta_prot_permin']:.1f}x  "
      f"(dilution dominates)")
print(f"m_ss  = {m_ss:.1f} nM")
print(f"P_ss  = {P_ss:.2f} uM monomer-equivalents")
print(f"D_ss  = {D_eq:.2f} uM dimer  (dimer fraction = {2*D_eq/P_ss:.3f})")
print(f"P_ss/2 = {P_ss/2:.2f} uM  <- perfect-dimer approximation")

## 4 · The two figures of the panel

Panel (a) integrates the equations up to 300 min. Panel (b) sweeps the dimer dissociation constant, $K_d$, to show the value beyond which the approximation $D = P/2$ ceases to hold.

In [ ]:
# Reproduces the two panels: (a) approach to steady state, (b) justification of D = P/2
t = np.linspace(0, 300, 601)
m_t, P_t = simulate_expression(p, t, t_on=30.0)

# Kd sweep for panel (b), at fixed P_tot = P_ss
Kd_grid = np.logspace(-3, 1.5, 200)
frac_dimer = np.array([2.0 * monomer_dimer_split(P_ss, kd)[1] / P_ss
                       for kd in Kd_grid])

fig, (axA, axB) = plt.subplots(1, 2, figsize=(11.0, 4.1))

# ---- panel a ----
axA.plot(t, m_t,       color="#c0392b", lw=1.9, label=r"mRNA $m$ (nM)")
axA.plot(t, P_t,       color="#17607a", lw=1.9, label=r"monomer $P$ (µM)")
axA.plot(t, P_t / 2.0, color="#d4a017", lw=1.7, ls="--",
         label=r"dimer $D = P/2$ (µM)")
axA.axvline(30.0, color="#8a9a95", lw=0.8, ls=":")
axA.annotate("induction  $t_{on}$ = 30 min", xy=(30.0, 22.0),
             xytext=(115.0, 27.0), fontsize=7.5, color="#5b6b66",
             arrowprops=dict(arrowstyle="->", color="#8a9a95", lw=0.8))
axA.set_xlabel("time (min)")
axA.set_ylabel("concentration")
axA.set_title("a   Expression reaches steady state", loc="left")
axA.legend(frameon=False, fontsize=8, loc="center right")

# ---- panel b ----
axB.plot(Kd_grid, frac_dimer, color="#17607a", lw=2.0)
axB.axvline(p["Kd_dim_uM"], color="#d4a017", lw=1.4, ls="--")
axB.text(p["Kd_dim_uM"] * 1.5, 0.55,
         f"Ni 1995: strong-dimer regime\n"
         f"$K_d$ = {p['Kd_dim_uM']} µM → fraction {2*D_eq/P_ss:.3f}",
         fontsize=7.5, color="#7a5c00")
axB.set_xscale("log")
axB.set_xlabel(r"dimerisation $K_d$ (µM)")
axB.set_ylabel(r"fraction of protein as dimer  $2D/P_{tot}$")
axB.set_title(r"b   Why $D = P/2$ is justified", loc="left")
axB.set_ylim(0.4, 1.02)

fig.tight_layout()
fig.savefig("M3_expression.png", dpi=300, bbox_inches="tight")
plt.show()
print("saved: M3_expression.png")

## 5 · Temperature sweep

The practical consequence of M3 is that lowering the temperature reduces the dimer peak. Because the assembly optimum of M4 lies at 2 µM, the culture temperature acts as a **control lever** on assembly yield and not only as a side effect.

In [ ]:
def scale_params_temperature(p, T_C, T_ref_C=37.0, Q10_growth=2.0, Q10_tx=2.0):
    """Rescale the temperature-dependent rates with a Q10 law.

    This is a coarse approximation: it predicts the sign and the order of
    magnitude of the effect of lowering the temperature to 30 or 25 C, but
    not a precise value.

    If the growth curve is measured at the three temperatures, Q10_growth is
    obtained from the data and ceases to be an assumption."""
    q = lambda Q10: Q10 ** ((T_C - T_ref_C) / 10.0)
    out = dict(p)
    out["t_double_min"]    = p["t_double_min"] / q(Q10_growth)
    out["alpha_tx_nM_min"] = p["alpha_tx_nM_min"] * q(Q10_tx)
    out["k_tl_permin"]     = p["k_tl_permin"] * q(Q10_tx)
    return out


t = np.linspace(0, 240, 481)
fig, ax = plt.subplots(figsize=(5.6, 4.0))
for T, c in [(37.0, "#c0392b"), (30.0, "#d4a017"), (25.0, "#17607a")]:
    _, P_T = simulate_expression(scale_params_temperature(p, T), t, t_on=30.0)
    D_T = P_T / 2.0
    ax.plot(t, D_T, color=c, lw=1.9, label=f"{T:.0f} °C")
    print(f"T = {T:4.1f} C -> D(240 min) = {D_T[-1]:5.2f} uM")

ax.axhline(2.0, color="#0b6e5a", lw=1.3, ls="--")
ax.text(236, 2.08, "M4 optimum: 2 µM", ha="right", fontsize=7.5,
        color="#0b6e5a")
ax.set_xlabel("time (min)")
ax.set_ylabel("coat-protein dimer (µM)")
ax.set_title("Temperature is a lever on the input to M4", loc="left")
ax.legend(frameon=False, title="culture")
fig.tight_layout()
fig.savefig("M3_temperature.png", dpi=300, bbox_inches="tight")
plt.show()

## Using the notebook

| To test... | Modify... |
|---|---|
| another promoter | `alpha_tx_nM_min` |
| another ribosome-binding site (RBS) | `k_tl_permin` |
| another culture medium | `t_double_min` |
| a later induction | `t_on` in `simulate_expression` |
| a weaker dimer | `Kd_dim_uM` |

**Once the RT-qPCR data are available**, the next step is to fit `alpha_tx_nM_min` and `t_half_mrna_min` to the measured curve (Mode B). The sampling design is important: a single end point cannot identify the two parameters separately, and about 7 points in the transient (2–180 min) are needed.